# Day 4 模块 1：模型评估指标

「好不好」不能只看感觉。今天把 **R²、MAE、MSE/RMSE** 放在同一套测试集上读。


In [ ]:
from pathlib import Path
import pandas as pd

candidate_paths = [
    Path('day4_cafe_sales.csv'),
    Path('day4') / 'day4_cafe_sales.csv',
    Path('教学课程') / 'day4' / 'day4_cafe_sales.csv',
]
for path in candidate_paths:
    if path.exists():
        csv_path = path
        break
else:
    raise FileNotFoundError('未找到 day4_cafe_sales.csv')

df = pd.read_csv(csv_path)
print(csv_path.resolve())
print('shape:', df.shape)
df.head()


In [ ]:
# 准备特征 X 和目标 y（与 Day 3 同一套，方便对比）
# 特征列：可能影响营收的输入
feature_cols = [
    'price', 'promotion', 'is_weekend', 'temp',
    'quality', 'competitors', 'holiday', 'location',
]
# 天气文字 → 数字分（晴最好，大雨最差）
weather_score_map = {'晴': 1.0, '多云': 0.8, '阴': 0.6, '小雨': 0.4, '大雨': 0.3}
df = df.copy()
df['weather_score'] = df['weather_label'].map(weather_score_map)

X = df[feature_cols + ['weather_score']].copy()  # 特征表
y = df['sales'].copy()  # 目标：营收

print('特征列:', list(X.columns))
print('样本数:', len(X))
X.head()


In [ ]:
from sklearn.model_selection import train_test_split

# test_size=0.2：约 20% 当测试集；random_state=42：随机种子，结果可复现
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print('训练集行数:', len(X_train))
print('测试集行数:', len(X_test))


## 1. 先训一个熟悉的随机森林

下面用 Day 3 同款参数，方便和昨天对照。


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=8,
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print('训练完成')


## 2. 三个读数，各管什么

| 指标 | 全称直觉 | 怎么读 | 好还是坏 |
| --- | --- | --- | --- |
| **R²** | 解释了多少营收波动 | 接近 **1** 更好；接近 **0** 几乎没比「每次猜平均」强 | 越大越好（也可为负） |
| **MAE** | 平均绝对误差 | 平均大概偏 **多少钱** | **越小越好** |
| **MSE** | 均方误差 | 把偏差平方再平均（大错被放大） | 越小越好 |
| **RMSE** | 均方根误差 | 对 MSE 开方，单位回到「钱」附近 | **越小越好** |

### R²（再钉一次）

- **不是**准确率，别说成「76% 猜对了」
- 测试集 R² 才是成绩单；训练分高一点常见

### MAE

```text
每一天 |预测 − 真实|，再对所有测试天求平均
```

人话：**平均大概偏多少元。**

### MSE 与 RMSE

```text
MSE  = 平均 (预测 − 真实)²
RMSE = √MSE
```

- 平方后，**偏得特别离谱的天**会拉高分数
- RMSE 开根号后，数量级又接近「钱」，方便和 MAE 一起看

| | MAE | RMSE |
| --- | --- | --- |
| 对大误差 | 较温和 | 更敏感 |
| 单位 | 钱 | 钱（同量级） |

今天：**会算、会读**即可，不要求推导证明。


In [ ]:
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = float(np.sqrt(mse))

print('测试集 R²  :', round(r2, 3))
print('测试集 MAE :', round(mae, 1), '（平均大概偏多少钱）')
print('测试集 MSE :', round(mse, 1))
print('测试集 RMSE:', round(rmse, 1), '（√MSE，对大误差更敏感）')


## 3. 对照表：真实 vs 预测（前几行）

总分看 R² / MAE；单行有时偏多、有时偏少，都正常。


In [ ]:
compare = pd.DataFrame({
    '真实营收': y_test.to_numpy()[:10],
    '预测营收': y_pred[:10].round(1),
})
compare['绝对偏差'] = (compare['预测营收'] - compare['真实营收']).abs().round(1)
compare


## 4. 小练习

1. 用自己的话写：R² 和 MAE 各回答什么问题？
2. 若 R² 很高但 MAE 也很大，可能说明什么？（提示：营收金额本身很大）
3. 为什么评估主要在**测试集**上算这些数？


In [ ]:
# 在这里写
